# Chapter 15: Valves and Flow Control

This notebook demonstrates valve modeling and flow control analysis using
NeqSim's `ThrottlingValve` class. We examine Cv-based valve sizing,
pressure drop characteristics, Joule–Thomson cooling, and the effect
of valve opening on flow control.

**Key concepts:**
- Cv / Kv valve sizing per IEC 60534
- Pressure drop vs. flow rate characteristic
- Joule–Thomson (JT) cooling effect through valves
- Valve opening control and turndown

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 15.1 Define Feed Stream and Throttling Valve

We set up a natural gas stream at high pressure and throttle it through
a control valve with a specified outlet pressure.

In [2]:
from neqsim import jneqsim

Stream = jneqsim.process.equipment.stream.Stream
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

# Create natural gas fluid
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 50.0, 150.0)
fluid.addComponent("methane", 0.85)
fluid.addComponent("ethane", 0.07)
fluid.addComponent("propane", 0.04)
fluid.addComponent("n-butane", 0.02)
fluid.addComponent("CO2", 0.015)
fluid.addComponent("nitrogen", 0.005)
fluid.setMixingRule("classic")

# Feed stream
feed = Stream("HP Gas", fluid)
feed.setFlowRate(50000.0, "kg/hr")
feed.setTemperature(50.0, "C")
feed.setPressure(150.0, "bara")

# Throttling valve
valve = ThrottlingValve("Choke Valve", feed)
valve.setOutletPressure(80.0, "bara")

# Build and run process
process = ProcessSystem()
process.add(feed)
process.add(valve)
process.run()

T_in = feed.getTemperature("C")
T_out = valve.getOutletStream().getTemperature("C")
P_in = feed.getPressure("bara")
P_out = valve.getOutletStream().getPressure("bara")
dP = valve.getDeltaPressure("bara")
Cv = valve.getCv("US")

print(f"Inlet:    {T_in:.1f} °C,  {P_in:.0f} bara")
print(f"Outlet:   {T_out:.1f} °C,  {P_out:.0f} bara")
print(f"ΔP:       {dP:.1f} bar")
print(f"JT cooling: {T_in - T_out:.1f} °C")
print(f"Cv (US):  {Cv:.1f}")

Inlet:    50.0 °C,  150 bara
Outlet:   27.0 °C,  80 bara
ΔP:       70.0 bar
JT cooling: 23.0 °C
Cv (US):  57.5


## 15.2 Pressure Drop vs. Flow Rate

At fixed inlet and outlet pressures, we vary the mass flow rate to observe
how the required Cv changes and how JT cooling varies.

In [3]:
flow_rates = np.linspace(10000, 80000, 15)  # kg/hr
cv_values = []
jt_cooling = []
outlet_temps = []

for flow in flow_rates:
    feed.setFlowRate(float(flow), "kg/hr")
    process.run()
    cv_values.append(valve.getCv("US"))
    T_out_i = valve.getOutletStream().getTemperature("C")
    outlet_temps.append(T_out_i)
    jt_cooling.append(T_in - T_out_i)

# Reset
feed.setFlowRate(50000.0, "kg/hr")
process.run()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(flow_rates / 1000, cv_values, 'b-o', linewidth=2, markersize=5)
ax1.set_xlabel('Mass Flow Rate [t/hr]', fontsize=12)
ax1.set_ylabel('Cv (US units)', fontsize=12)
ax1.set_title('Required Valve Cv vs. Flow Rate', fontsize=13)
ax1.grid(True, alpha=0.3)

ax2.plot(flow_rates / 1000, jt_cooling, 'r-s', linewidth=2, markersize=5)
ax2.set_xlabel('Mass Flow Rate [t/hr]', fontsize=12)
ax2.set_ylabel('JT Cooling [°C]', fontsize=12)
ax2.set_title('Joule–Thomson Cooling vs. Flow Rate', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch15_cv_and_jt_vs_flow.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ch15_cv_and_jt_vs_flow.png")

Figure saved: ch15_cv_and_jt_vs_flow.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_34952\1933055561.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 15.3 JT Cooling Across Different Pressure Drops

We fix the flow rate and vary the valve outlet pressure to map the
relationship between pressure drop and temperature drop (JT effect).

In [4]:
outlet_pressures = np.linspace(30.0, 140.0, 20)
delta_T = []
delta_P = []

for p_out in outlet_pressures:
    valve.setOutletPressure(float(p_out), "bara")
    process.run()
    T_outlet = valve.getOutletStream().getTemperature("C")
    delta_T.append(T_in - T_outlet)
    delta_P.append(P_in - float(p_out))

# Reset
valve.setOutletPressure(80.0, "bara")
process.run()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(delta_P, delta_T, 'g-o', linewidth=2.5, markersize=5)
ax.set_xlabel('Pressure Drop ΔP [bar]', fontsize=12)
ax.set_ylabel('Temperature Drop ΔT [°C]', fontsize=12)
ax.set_title('Joule–Thomson Cooling vs. Pressure Drop (Natural Gas at 150 bara)', fontsize=13)
ax.grid(True, alpha=0.3)

# Annotate approximate JT coefficient
if len(delta_P) > 2 and delta_P[-1] > delta_P[0]:
    jt_coeff = (delta_T[-1] - delta_T[0]) / (delta_P[-1] - delta_P[0])
    ax.annotate(f'Avg. JT coeff ≈ {jt_coeff:.2f} °C/bar',
                xy=(delta_P[len(delta_P)//2], delta_T[len(delta_T)//2]),
                fontsize=11, fontweight='bold',
                xytext=(20, 20), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='black'))

plt.tight_layout()
plt.savefig("../figures/ch15_jt_cooling_vs_dp.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ch15_jt_cooling_vs_dp.png")

Figure saved: ch15_jt_cooling_vs_dp.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_34952\2778278825.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 15.4 Pressure and Temperature Profiles Through the Valve

We summarize the inlet and outlet conditions and visualize the
pressure-temperature path through the throttling process.

In [5]:
# Compute the P-T path through the valve for several outlet pressures
p_out_range = np.linspace(40.0, 145.0, 25)
T_path = []
P_path = []

for p_out in p_out_range:
    valve.setOutletPressure(float(p_out), "bara")
    process.run()
    T_path.append(valve.getOutletStream().getTemperature("C"))
    P_path.append(float(p_out))

# Reset
valve.setOutletPressure(80.0, "bara")
process.run()

fig, ax = plt.subplots(figsize=(10, 6))

# Plot P-T path
ax.plot(T_path, P_path, 'b-o', linewidth=2, markersize=4, label='Isenthalpic expansion')
ax.plot(T_in, P_in, 'r*', markersize=18, zorder=5, label=f'Inlet ({T_in:.0f} °C, {P_in:.0f} bara)')

# Mark design outlet
ax.plot(valve.getOutletStream().getTemperature("C"), 80.0, 'gs', markersize=12, zorder=5,
        label=f'Design outlet (80 bara)')

ax.set_xlabel('Temperature [°C]', fontsize=12)
ax.set_ylabel('Pressure [bara]', fontsize=12)
ax.set_title('P-T Path Through Throttling Valve (Isenthalpic)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch15_pt_path.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ch15_pt_path.png")

Figure saved: ch15_pt_path.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_34952\3665367727.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 15.5 Valve Opening and Cv Relationship

We examine how effective Cv changes with valve percent opening,
demonstrating valve rangeability and control characteristics.

In [6]:
# Set a known Cv for the valve
valve.setCv(200.0, "US")

openings = np.linspace(10, 100, 10)
effective_cv = []
outlet_T = []
outlet_P = []

for opening in openings:
    valve.setPercentValveOpening(float(opening))
    process.run()
    effective_cv.append(valve.getCv("US") * float(opening) / 100.0)  # effective Cv
    outlet_T.append(valve.getOutletStream().getTemperature("C"))
    outlet_P.append(valve.getOutletStream().getPressure("bara"))

# Reset
valve.setPercentValveOpening(100.0)
valve.setOutletPressure(80.0, "bara")
process.run()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(openings, effective_cv, 'b-o', linewidth=2, markersize=5)
ax1.set_xlabel('Valve Opening [%]', fontsize=12)
ax1.set_ylabel('Effective Cv (US)', fontsize=12)
ax1.set_title('Effective Cv vs. Valve Opening', fontsize=13)
ax1.grid(True, alpha=0.3)

ax2.plot(openings, outlet_P, 'm-s', linewidth=2, markersize=5, label='Outlet Pressure')
ax2_twin = ax2.twinx()
ax2_twin.plot(openings, outlet_T, 'r-^', linewidth=2, markersize=5, label='Outlet Temp')

ax2.set_xlabel('Valve Opening [%]', fontsize=12)
ax2.set_ylabel('Outlet Pressure [bara]', fontsize=12, color='m')
ax2_twin.set_ylabel('Outlet Temperature [°C]', fontsize=12, color='r')
ax2.set_title('Outlet Conditions vs. Valve Opening', fontsize=13)
ax2.grid(True, alpha=0.3)

# Combined legend
lines_1, labels_1 = ax2.get_legend_handles_labels()
lines_2, labels_2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best', fontsize=10)

plt.tight_layout()
plt.savefig("../figures/ch15_valve_opening.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ch15_valve_opening.png")

Figure saved: ch15_valve_opening.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_34952\1806246903.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Discussion

The throttling valve is an isenthalpic process: the fluid enthalpy remains
constant while pressure drops. For real gases (especially at high pressure),
this causes a measurable temperature drop known as the Joule–Thomson effect.

**Key observations:**

1. **JT cooling** is approximately proportional to pressure drop for modest
   ΔP values. The JT coefficient for natural gas at typical wellhead conditions
   is roughly 0.3–0.5 °C/bar. This cooling is critical in production systems
   because it can bring conditions close to hydrate formation.

2. **Cv sizing** follows the ISA/IEC 60534 standard. The required Cv increases
   approximately with the square root of pressure drop for gas service. Oversizing
   the valve leads to poor controllability at low openings, while undersizing
   limits maximum throughput.

3. **Valve rangeability** — the ratio of maximum to minimum controllable flow —
   is a key design parameter. Equal-percentage valve characteristics are preferred
   for gas service because they provide more linear installed gain.

4. **Production optimization** implications: choke valves at wellheads control
   production rate and downstream pressure. Optimal choke settings balance
   production rate against hydrate risk, sand erosion, and downstream process
   requirements.